In [54]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bryanpark/sudoku")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\Playdata\.cache\kagglehub\datasets\bryanpark\sudoku\versions\3


In [75]:
import copy
import sudoku
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


solution = sudoku.construct_puzzle_solution()
puzzle, givens = sudoku.pluck(copy.deepcopy(solution), n=30)
print(puzzle)
sudoku.display(puzzle)
print("givens:", givens)

[[1, 5, 3, 0, 0, 0, 8, 4, 0], [0, 0, 8, 0, 0, 0, 6, 0, 9], [0, 7, 6, 0, 0, 4, 0, 2, 0], [0, 8, 0, 0, 0, 6, 0, 1, 2], [0, 0, 0, 0, 0, 5, 0, 0, 0], [6, 0, 0, 9, 2, 7, 3, 0, 8], [0, 4, 0, 0, 1, 2, 0, 0, 3], [0, 3, 7, 5, 9, 0, 1, 6, 0], [0, 0, 1, 0, 7, 0, 0, 9, 5]]
1 5 3 _ _ _ 8 4 _
_ _ 8 _ _ _ 6 _ 9
_ 7 6 _ _ 4 _ 2 _
_ 8 _ _ _ 6 _ 1 2
_ _ _ _ _ 5 _ _ _
6 _ _ 9 2 7 3 _ 8
_ 4 _ _ 1 2 _ _ 3
_ 3 7 5 9 _ 1 6 _
_ _ 1 _ 7 _ _ 9 5
givens: 37


In [56]:
import numpy as np
quizzes = np.zeros((1000000, 81), np.int32)
solutions = np.zeros((1000000, 81), np.int32)
for i, line in enumerate(open('data/sudoku.csv', 'r').read().splitlines()[1:]):
    quiz, solution = line.split(",")
    for j, q_s in enumerate(zip(quiz, solution)):
        q, s = q_s
        quizzes[i, j] = q
        solutions[i, j] = s
quizzes = quizzes.reshape((-1, 9, 9))
solutions = solutions.reshape((-1, 9, 9))

In [57]:
print(quizzes)

[[[0 0 4 ... 2 0 9]
  [0 0 5 ... 0 0 1]
  [0 7 0 ... 0 4 3]
  ...
  [6 0 0 ... 1 0 5]
  [0 0 3 ... 6 9 0]
  [0 4 2 ... 3 0 0]]

 [[0 4 0 ... 0 5 0]
  [1 0 7 ... 9 6 0]
  [5 2 0 ... 0 0 0]
  ...
  [0 9 0 ... 5 4 3]
  [6 0 0 ... 7 0 0]
  [2 5 0 ... 1 0 0]]

 [[6 0 0 ... 3 8 4]
  [0 0 8 ... 0 7 2]
  [0 0 0 ... 0 0 5]
  ...
  [3 1 0 ... 0 5 0]
  [0 8 9 ... 0 0 0]
  [5 0 2 ... 1 9 0]]

 ...

 [[0 0 0 ... 8 2 0]
  [0 6 1 ... 0 3 0]
  [0 5 0 ... 0 0 0]
  ...
  [0 0 7 ... 0 6 5]
  [0 0 0 ... 4 0 8]
  [0 8 6 ... 0 0 0]]

 [[0 7 0 ... 6 9 0]
  [0 0 3 ... 0 0 1]
  [0 0 0 ... 0 2 0]
  ...
  [0 0 0 ... 0 4 0]
  [0 5 1 ... 9 0 0]
  [9 4 0 ... 0 0 7]]

 [[3 0 0 ... 6 2 0]
  [1 0 0 ... 4 0 0]
  [0 0 5 ... 8 3 0]
  ...
  [4 8 0 ... 0 1 0]
  [2 0 3 ... 0 0 0]
  [0 7 0 ... 0 9 0]]]


In [58]:
df = pd.read_csv('data/sudoku.csv',dtype=str)
df

,quizzes,solutions
0,0043002090050090010700600430060020871900074000...,8643712593258497619712658434361925871986574322...
1,0401000501070039605200080000000000170009068008...,3461792581875239645296483719658324174729168358...
2,6001203840084590720000060050002640300700800069...,6951273841384596727248369158512647392739815469...
3,4972000001004000050000160986203000403009000000...,4972583161864397252537164986293815473759641828...
4,0059103080094030600275001000300002010008200070...,4659123781894735623275681497386452919548216372...
...,...,...
999995,3000280000290000300054001077402030980086070031...,3175289464291768356854391277462135989586472131...
999996,0030006000040860057000009409350407208067200502...,5234976811942863757685139429356417288167294532...
999997,0003508200618040300500090000700600029030070100...,7493568212618745393582197468749613529235876146...
999998,0702006900030400010000650205600300000947005800...,4752816936239478511893657245628341793947165828...


In [59]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 2 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   quizzes    1000000 non-null  object
 1   solutions  1000000 non-null  object
dtypes: object(2)
memory usage: 15.3+ MB


In [60]:
# 81자리가 아닌 문제 확인 > 없음
df[df['quizzes'].str.len()!=81]

,quizzes,solutions


In [61]:
# 81자리가 아닌 답안지 확인 > 없음
df[df['solutions'].str.len()!=81]

,quizzes,solutions


입력(quiz) :  0~9 (0은 빈칸을 뜻함)  
정답(solution) : 1~9  
학습 라벨 : (solution - 1) → 0~8

In [ ]:
quiz_str = df['quizzes'][0]
quiz = np.array([int(c) for c in quiz_str]).reshape(9,9).astype(np.int64)
quiz

array([[0, 0, 4, 3, 0, 0, 2, 0, 9],
       [0, 0, 5, 0, 0, 9, 0, 0, 1],
       [0, 7, 0, 0, 6, 0, 0, 4, 3],
       [0, 0, 6, 0, 0, 2, 0, 8, 7],
       [1, 9, 0, 0, 0, 7, 4, 0, 0],
       [0, 5, 0, 0, 8, 3, 0, 0, 0],
       [6, 0, 0, 0, 0, 0, 1, 0, 5],
       [0, 0, 3, 5, 0, 8, 6, 9, 0],
       [0, 4, 2, 9, 1, 0, 3, 0, 0]])

In [ ]:


class SudokuDataset(Dataset):
    """
    반환:
      X: (10, 9, 9) float32 one-hot (0~9)
      y: (9, 9) int64  (0~8)  # 정답 1~9를 0~8로 shift
      mask: (9, 9) bool  # quiz==0 (빈칸 위치)
    """
    def __init__(self, df: pd.DataFrame, quiz_col: str, sol_col: str):
        self.quiz = df[quiz_col].values
        self.sol  = df[sol_col].values

    def __len__(self):
        return len(self.quiz)

    @staticmethod
    def _str_to_grid(s: str) -> np.ndarray:

        b = np.frombuffer(s.encode("ascii"), dtype=np.uint8) - ord("0")
        return b.reshape(9, 9).astype(np.int64)

    def __getitem__(self, idx: int):
        quiz_str = self.quiz[idx]
        sol_str  = self.sol[idx]

        quiz = self._str_to_grid(quiz_str)     # (9,9) 0~9
        sol  = self._str_to_grid(sol_str)      # (9,9) 1~9

        mask = (quiz == 0)                                  # 마스크: 빈칸 위치만 학습,평가에 쓰는 용도

        # 라벨: 1~9 -> 0~8
        y = sol - 1

        # 입력 원핫: 0~9 (10클래스)
        # X[d, i, j] = 1 if quiz[i,j] == d
        X = np.zeros((10, 9, 9), dtype=np.float32)
        for d in range(10):
            X[d] = (quiz == d)

        # torch 텐서로 변환
        X = torch.from_numpy(X)                 # float32
        y = torch.from_numpy(y.astype(np.int64))# int64
        mask = torch.from_numpy(mask)           # bool

        return X, y, mask

In [ ]:
dataset = SudokuDataset(df, "quizzes", "solutions")

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,      # 윈도우면 0~2부터 시작 추천
    pin_memory=True
)

X, y, mask = next(iter(loader))
print("X:", X.shape, X.dtype)           # (B, 10, 9, 9) torch.float32
print("y:", y.shape, y.dtype)           # (B, 9, 9) torch.int64
print("mask:", mask.shape, mask.dtype)  # (B, 9, 9) torch.bool
print("blank ratio in batch:", mask.float().mean().item())

X: torch.Size([64, 10, 9, 9]) torch.float32
y: torch.Size([64, 9, 9]) torch.int64
mask: torch.Size([64, 9, 9]) torch.bool
blank ratio in batch: 0.5835262537002563


c:\Users\Playdata\deep_learning\dl_venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


10

In [ ]:
model = nn.Sequential(
    nn.Conv2d(X[0].size(0)),         # 왜 Conv2d  
    nn.ReLU(),
)

TypeError: Conv2d.__init__() missing 2 required positional arguments: 'out_channels' and 'kernel_size'

## conv2d  
작은 필터(예: 3×3)를  
전체 격자에 공유해서  
이웃 패턴을 인식  
주변 칸을 보면서 규칙을 학습하는 연산
